In [1]:
import pandas as pd
import ast
from collections import Counter

# Load both datasets
med_df   = pd.read_csv('../datasets/stage1/medications.csv')
drugs_df = pd.read_csv(
    '../datasets/stage2/drugs_side_effects_drugs_com.csv')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# AUTO-LOOKUP: 22 diseases found in drugs_side_effects
# Pulls top reviewed drug per condition automatically
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONDITION_MAP = {
    'AIDS'                         : 'AIDS/HIV',
    'Acne'                         : 'Acne',
    'Allergy'                      : 'Allergies',
    'Arthritis'                    : 'Rheumatoid Arthritis',
    'Bronchial Asthma'             : 'Asthma',
    'Cervical spondylosis'         : 'Pain',
    'Common Cold'                  : 'Colds & Flu',
    'Diabetes'                     : 'Diabetes (Type 2)',
    'GERD'                         : 'GERD (Heartburn)',
    'Gastroenteritis'              : 'Gastrointestinal',
    'Heart attack'                 : 'Angina',
    'Hypertension'                 : 'Hypertension',
    'Hypothyroidism'               : 'Hypothyroidism',
    'Hypoglycemia'                 : 'Diabetes (Type 1)',
    'Migraine'                     : 'Migraine',
    'Osteoarthristis'              : 'Osteoarthritis',
    'Paralysis (brain hemorrhage)' : 'Stroke',
    'Pneumonia'                    : 'Pneumonia',
    'Psoriasis'                    : 'Psoriasis',
    'Urinary tract infection'      : 'UTI',
    'Fungal infection'             : 'Eczema',
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FALLBACK: 19 diseases NOT in drugs_side_effects
# WHO/clinical standard first-line drugs
# These never change — stable reference
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FALLBACK = {
    'Dengue'                                : 'acetaminophen',
    'Malaria'                               : 'chloroquine',
    'Typhoid'                               : 'ciprofloxacin',
    'Tuberculosis'                          : 'isoniazid',
    'Chicken pox'                           : 'acyclovir',
    'Impetigo'                              : 'mupirocin',
    'hepatitis A'                           : 'supportive_care',
    'Hepatitis B'                           : 'entecavir',
    'Hepatitis C'                           : 'sofosbuvir',
    'Hepatitis D'                           : 'peginterferon alfa',
    'Hepatitis E'                           : 'ribavirin',
    'Alcoholic hepatitis'                   : 'prednisolone',
    'Jaundice'                              : 'ursodiol',
    'Chronic cholestasis'                   : 'cholestyramine',
    'Hyperthyroidism'                       : 'methimazole',
    'Varicose veins'                        : 'diosmin',
    'Dimorphic hemmorhoids(piles)'          : 'hydrocortisone',
    '(vertigo) Paroymsal Positional Vertigo': 'meclizine',
    'Drug Reaction'                         : 'diphenhydramine',
    'Peptic ulcer disease'                  : 'amoxicillin',
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Apply IN MEMORY — medications.csv never touched
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Building drug mapping IN MEMORY:")
print("="*65)

for idx, row in med_df.iterrows():
    disease = row['Disease'].strip()

    if disease in CONDITION_MAP:
        cond = CONDITION_MAP[disease]
        drug = (drugs_df[
            drugs_df['medical_condition'] == cond]
            .sort_values('no_of_reviews', ascending=False)
            ['drug_name'].iloc[0])
        med_df.at[idx, 'Medication'] = str([drug])
        print(f"  ✅ {disease:<45} → {drug} [auto]")

    elif disease in FALLBACK:
        drug = FALLBACK[disease]
        med_df.at[idx, 'Medication'] = str([drug])
        print(f"  ✅ {disease:<45} → {drug} [clinical]")

# Verify zero duplicates
drugs_list   = []
for _, row in med_df.iterrows():
    try:    drugs_list.append(
                ast.literal_eval(row['Medication'])[0])
    except: drugs_list.append(str(row['Medication']))

dupes = {k:v for k,v in Counter(drugs_list).items()
         if v > 1}

print(f"\nTotal diseases  : {len(med_df)}")
print(f"Unique drugs    : {len(set(drugs_list))}")
if dupes:
    print(f"⚠️  Duplicates  : {dupes}")
else:
    print(f"✅ Zero duplicates — 41 unique drugs!")
print(f"\nIN MEMORY only — medications.csv untouched ✅")

Building drug mapping IN MEMORY:
  ✅ Fungal infection                              → dupilumab [auto]
  ✅ Allergy                                       → levocetirizine [auto]
  ✅ GERD                                          → omeprazole [auto]
  ✅ Chronic cholestasis                           → cholestyramine [clinical]
  ✅ Drug Reaction                                 → diphenhydramine [clinical]
  ✅ Peptic ulcer disease                          → amoxicillin [clinical]
  ✅ AIDS                                          → Biktarvy [auto]
  ✅ Diabetes                                      → dulaglutide [auto]
  ✅ Gastroenteritis                               → Entereg [auto]
  ✅ Bronchial Asthma                              → montelukast [auto]
  ✅ Hypertension                                  → lisinopril [auto]
  ✅ Migraine                                      → sumatriptan [auto]
  ✅ Cervical spondylosis                          → acetaminophen / hydrocodone [auto]
  ✅ Paralysis (br

In [2]:
import pandas as pd
import numpy as np
import warnings
from rapidfuzz import process, fuzz
warnings.filterwarnings('ignore')

# Load training data
train = pd.read_csv('../datasets/stage1/Training.csv')

# med_df already fixed IN MEMORY from Cell 1
# Just use med_df directly — do NOT reload from CSV

# Fuzzy fix typos in Training.csv
# Same approach as Stage 1
valid_diseases = list(med_df['Disease'].str.strip().unique())

def fuzzy_fix(name, valid, threshold=90):
    name_c = name.strip().lower()
    result = process.extractOne(
        name_c,
        [v.lower() for v in valid],
        scorer=fuzz.ratio
    )
    if result and result[1] >= threshold:
        idx = [v.lower() for v in valid].index(result[0])
        return valid[idx]
    return name

train['prognosis'] = train['prognosis'].apply(
    lambda x: fuzzy_fix(x, valid_diseases))

# Merge Training.csv with med_df (already fixed in Cell 1)
merged = train.merge(
    med_df,
    left_on='prognosis',
    right_on='Disease'
)

print(f"Training rows     : {len(train)}")
print(f"Medications rows  : {len(med_df)}")
print(f"Merged rows       : {len(merged)}")

if len(merged) == len(train):
    print("✅ All rows matched — Zero mismatch!")
else:
    print(f"⚠️ Missing: {len(train) - len(merged)} rows")

print(f"\nSample disease → drug mapping:")
print(merged[['prognosis','Medication']].drop_duplicates().head(10))


Training rows     : 4920
Medications rows  : 41
Merged rows       : 4920
✅ All rows matched — Zero mismatch!

Sample disease → drug mapping:
               prognosis           Medication
0       Fungal infection        ['dupilumab']
10               Allergy   ['levocetirizine']
20                  GERD       ['omeprazole']
30   Chronic cholestasis   ['cholestyramine']
40         Drug Reaction  ['diphenhydramine']
50  Peptic ulcer disease      ['amoxicillin']
60                  AIDS         ['Biktarvy']
70              Diabetes      ['dulaglutide']
80       Gastroenteritis          ['Entereg']
90      Bronchial Asthma      ['montelukast']


In [3]:
# Find which diseases got dropped
train_diseases = set(train['prognosis'].unique())
med_diseases = set(med_df['Disease'].unique())
missing = train_diseases - med_diseases

print(f"Missing count: {len(missing)}")
print(f"\nDiseases in training but NOT in medications:")
for d in missing:
    print(f"  '{d}'")

print(f"\nDiseases in medications but NOT in training:")
extra = med_diseases - train_diseases
for d in extra:
    print(f"  '{d}'")

Missing count: 0

Diseases in training but NOT in medications:

Diseases in medications but NOT in training:


In [4]:
# Fix trailing spaces in training data
train['prognosis'] = train['prognosis'].str.strip()

# Merge again after fix
merged = train.merge(
    med_df,
    left_on='prognosis',
    right_on='Disease'
)

print(f"Training rows  : {len(train)}")
print(f"Merged rows    : {len(merged)}")

if len(merged) == len(train):
    print("✅ All rows matched — Zero mismatch!")
else:
    print(f"⚠️ Still missing: {len(train) - len(merged)} rows")

Training rows  : 4920
Merged rows    : 4920
✅ All rows matched — Zero mismatch!


In [5]:
import ast

# Fix medication column — convert string list to first drug
def get_first_drug(med_string):
    try:
        # Convert string "['Drug1', 'Drug2']" to actual list
        med_list = ast.literal_eval(med_string)
        return med_list[0]  # Return first drug as target
    except:
        return str(med_string)

# Apply fix
merged['primary_drug'] = merged['Medication'].apply(get_first_drug)

print("Sample medications:")
print(merged[['prognosis', 'primary_drug']].drop_duplicates().head(10))
print(f"\nTotal unique drugs: {merged['primary_drug'].nunique()}")
print(f"Total unique diseases: {merged['prognosis'].nunique()}")

Sample medications:
               prognosis     primary_drug
0       Fungal infection        dupilumab
10               Allergy   levocetirizine
20                  GERD       omeprazole
30   Chronic cholestasis   cholestyramine
40         Drug Reaction  diphenhydramine
50  Peptic ulcer disease      amoxicillin
60                  AIDS         Biktarvy
70              Diabetes      dulaglutide
80       Gastroenteritis          Entereg
90      Bronchial Asthma      montelukast

Total unique drugs: 41
Total unique diseases: 41


In [6]:
from sklearn.preprocessing import LabelEncoder

# X = symptom features (same as Stage 1)
X = merged.drop(columns=['prognosis', 'Disease', 'Medication', 'primary_drug'])

# y = drug to recommend
le_drug = LabelEncoder()
y = le_drug.fit_transform(merged['primary_drug'])

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"\nDrug encoding sample:")
for drug, code in zip(le_drug.classes_[:5], range(5)):
    print(f"  {drug} → {code}")

X shape : (4920, 132)
y shape : (4920,)

Drug encoding sample:
  Biktarvy → 0
  Entereg → 1
  acetaminophen → 2
  acetaminophen / hydrocodone → 3
  acyclovir → 4


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

# Train Random Forest
print("\nTraining Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("Training complete ✅")

# Evaluate
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nRandom Forest Accuracy : {accuracy*100:.2f}%")
print(f"Paper 2 RF Accuracy    : 91.00%")
print(f"Your improvement       : {(accuracy-0.91)*100:+.2f}%")

Training samples : 3936
Testing samples  : 984

Training Random Forest...
Training complete ✅

Random Forest Accuracy : 100.00%
Paper 2 RF Accuracy    : 91.00%
Your improvement       : +9.00%


In [8]:
# Confidence score for drug recommendation
y_proba = rf.predict_proba(X_test)
confidence = y_proba.max(axis=1)

print("DRUG RECOMMENDATION CONFIDENCE")
print("="*40)
print(f"Average confidence : {confidence.mean()*100:.2f}%")
print(f"Minimum confidence : {confidence.min()*100:.2f}%")
print(f"Maximum confidence : {confidence.max()*100:.2f}%")

high   = (confidence >= 0.85).sum()
medium = ((confidence >= 0.70) & (confidence < 0.85)).sum()
low    = (confidence < 0.70).sum()

print(f"\nConfidence Tiers:")
print(f"  ✅ HIGH   (>=85%) : {high} predictions")
print(f"  ⚠️  MEDIUM (70-85%): {medium} predictions")
print(f"  🚨 LOW    (<70%)  : {low} predictions")

DRUG RECOMMENDATION CONFIDENCE
Average confidence : 50.55%
Minimum confidence : 13.90%
Maximum confidence : 86.86%

Confidence Tiers:
  ✅ HIGH   (>=85%) : 21 predictions
  ⚠️  MEDIUM (70-85%): 155 predictions
  🚨 LOW    (<70%)  : 808 predictions


In [9]:
import numpy as np
import lime                        # ← ADD THIS
import lime.lime_tabular           # ← ADD THIS
X_train_clean = np.nan_to_num(X_train.values, nan=0.0)
X_test_clean  = np.nan_to_num(X_test.values,  nan=0.0)
# Wrap predict_proba to handle NaN internally
def safe_predict_proba(data):
    data_clean = np.nan_to_num(data, nan=0.0)
    return rf.predict_proba(data_clean)

# Recreate LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_clean,
    feature_names=list(X.columns),
    class_names=le_drug.classes_,
    mode='classification',
    discretize_continuous=False,
    random_state=42
)

# Explain first test prediction
patient_idx = 0
exp = lime_explainer.explain_instance(
    X_test_clean[patient_idx],
    safe_predict_proba,
    num_features=5,
    top_labels=1
)

pred_drug_idx = rf.predict(
    X_test_clean[patient_idx].reshape(1,-1))[0]
pred_drug = le_drug.inverse_transform([pred_drug_idx])[0]
pred_conf = rf.predict_proba(
    X_test_clean[patient_idx].reshape(1,-1)).max()

print(f"Recommended Drug : {pred_drug}")
print(f"Confidence       : {pred_conf*100:.2f}%")
print(f"\nLIME Explanation — Top 5 reasons:")
print("="*45)
top_label = exp.top_labels[0]
for feature, weight in exp.as_list(label=top_label):
    direction = "supports ✅" if weight > 0 else "against ❌"
    print(f"  {feature[:35]:<35} {direction} ({weight:+.3f})")

Recommended Drug : levothyroxine
Confidence       : 81.40%

LIME Explanation — Top 5 reasons:
  enlarged_thyroid                    supports ✅ (+0.013)
  puffy_face_and_eyes                 supports ✅ (+0.013)
  weight_gain                         supports ✅ (+0.013)
  swollen_extremeties                 supports ✅ (+0.013)
  cold_hands_and_feets                supports ✅ (+0.012)


In [10]:
# ── MLP Neural Network Drug Recommendation ──
# Addresses Paper 1 (Siva Sai) stated limitation:
# "Sophisticated algorithms based on neural networks 
#  can also be experimented out."

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Step 1: Scale features (MLP needs scaling, RF does not)
scaler = StandardScaler()

X_train_clean = np.nan_to_num(X_train.values, nan=0.0)
X_test_clean  = np.nan_to_num(X_test.values,  nan=0.0)

X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled  = scaler.transform(X_test_clean)

# Step 2: Train MLP
print("Training MLP Neural Network...")
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
mlp.fit(X_train_scaled, y_train)
print("Training complete ✅")

# Step 3: Evaluate and Compare
y_pred_mlp = mlp.predict(X_test_scaled)
y_pred_rf  = rf.predict(X_test_clean)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
acc_rf  = accuracy_score(y_test, y_pred_rf)

print("\n" + "="*50)
print("MODEL COMPARISON — Drug Recommendation")
print("="*50)
print(f"Paper 1 RF baseline     : 91.00%")
print(f"Your Random Forest      : {acc_rf*100:.2f}%")
print(f"Your MLP Neural Network : {acc_mlp*100:.2f}%")

if acc_mlp > 0.91:
    print(f"\n✅ MLP beats Paper 1 RF baseline")
if acc_mlp > acc_rf:
    print(f"✅ MLP beats your own RF — use MLP as final model")
else:
    print(f"✅ RF still better — RF remains final model")
    print(f"✅ But MLP addresses Paper 1 stated limitation")

Training MLP Neural Network...
Training complete ✅

MODEL COMPARISON — Drug Recommendation
Paper 1 RF baseline     : 91.00%
Your Random Forest      : 100.00%
Your MLP Neural Network : 100.00%

✅ MLP beats Paper 1 RF baseline
✅ RF still better — RF remains final model
✅ But MLP addresses Paper 1 stated limitation


In [11]:
import joblib
import os

save_dir = '../models_trained'

# Save Random Forest model
joblib.dump(rf, f'{save_dir}/rf_drug_model.pkl')

# Save drug label encoder
joblib.dump(le_drug, f'{save_dir}/le_drug.pkl')

# Save clean training data for LIME
np.save(f'{save_dir}/X_train_clean.npy', X_train_clean)

# Save feature names
joblib.dump(list(X.columns), f'{save_dir}/drug_feature_names.pkl')
joblib.dump(mlp,    f'{save_dir}/mlp_drug_model.pkl')
joblib.dump(scaler, f'{save_dir}/drug_scaler.pkl')

print("MLP model saved ✅")
print("Stage 2 models saved ✅")
print(f"\nSaved files:")
for f in os.listdir(save_dir):
    print(f"  {f}")

MLP model saved ✅
Stage 2 models saved ✅

Saved files:
  adr_severity_dict.pkl
  chain_adr_model.pkl
  chain_order.pkl
  disease_mapping.pkl
  drug_class_bridge.pkl
  drug_feature_names.pkl
  drug_scaler.pkl
  ensemble_adr_models.pkl
  feature_names.pkl
  hf_map.pkl
  label_encoder.pkl
  le_adr_ac.pkl
  le_adr_cc.pkl
  le_adr_drug.pkl
  le_adr_tc.pkl
  le_drug.pkl
  mlb_adr.pkl
  mlp_drug_model.pkl
  rf_drug_model.pkl
  severity_dict.pkl
  stage3_adr_labels.pkl
  stage3_feature_cols.pkl
  xgb_disease_model.pkl
  X_test.csv
  X_train_clean.npy
